In [1]:
import csv

with open('interop_orig.csv', 'r') as csvfile:
    reader = csv.DictReader(csvfile)
    with open('harman_clean.csv','w') as write_file:
        fieldname = ['TimeStamp','lat','long','yawRate']
        csv_writer = csv.DictWriter(write_file,fieldnames=fieldname)
        csv_writer.writeheader()
        for line in reader:
            filtered_line = {col:line[col] for col in fieldname}

            csv_writer.writerow(filtered_line)


In [2]:
from datetime import datetime
import pandas as pd
from pydantic import BaseModel


class Point(BaseModel):
    latitude: float
    longitude: float
    timestamp: float  # Unix timestamp
    yaw_rate: float = 0.0   # Optional yaw rate, default to 0.0

def extract_points_from_csv(file_path: str, lat_col: str = "lat", lon_col: str = "long", time_col: str = "TimeStamp", yaw_col: str = "yawRate") -> list[Point]:
    df = pd.read_csv(file_path)
    # df = df.iloc[::25]
    points = []
    # Use today's date or set a fixed base date
    base_date = datetime(2025, 2, 21).date()
    
    for _, row in df.iterrows():
        try:
            # Combine date and time
            full_datetime_str = f"{base_date} {row[time_col]}"
            full_datetime = datetime.strptime(full_datetime_str, "%Y-%m-%d %H:%M:%S.%f")
            timestamp = full_datetime.timestamp()
            yawRate = float(row[yaw_col]) if yaw_col in row else 0.0
            lat = float(row[lat_col])
            long = float(row[lon_col])
            # yawRate = float(row[yaw_col])
            point = Point(latitude=lat, longitude=long, timestamp=timestamp, yaw_rate=yawRate)
            points.append(point)
        except (ValueError, TypeError, KeyError):
            continue  # skip rows with invalid data

    return points

# Example usage
points = extract_points_from_csv("harman_clean.csv")

print(len(points))


2591


In [3]:
import folium

# Create the map centered at the first point
fmap = folium.Map(location=[points[0].latitude, points[0].longitude], zoom_start=16)

batch_size = 100
for i in range(0, len(points), batch_size):
    batch_points = points[i:i + batch_size]
    if len(batch_points) < 2:
        continue
    points_portion = batch_points  # or use slicing if needed

    # Add original points (purple) for this batch
    for p in points_portion:
        folium.CircleMarker(
            location=[p.latitude, p.longitude],
            radius=2,
            color='purple',
            fill=True,
            fill_color='purple'
        ).add_to(fmap)

fmap

/Users/hpshigli/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
# service = 'match'
# version = 'v1'
# profile = 'driving'
# host = 'http://localhost:5001'

In [5]:
print("Type of points:", type(points))
try:
    print("Length of points:", len(points))
except Exception as e:
    print("Error when checking length:", e)

Type of points: <class 'list'>
Length of points: 2591


In [6]:
# # The variable 'response' is already defined in previous cells.
# if response.status_code == 200:
#     data = response.json()
#     route_coordinates = data['matchings'][0]['geometry']['coordinates']
#     matched_route = data['matchings'][0]

In [7]:
import requests
import folium

fmap = folium.Map(location=[points[0].latitude, points[0].longitude], zoom_start=16)
BATCH_SIZE = 100

print(f"Total batches expected: {(len(points) + BATCH_SIZE - 1) // BATCH_SIZE}")

for start in range(0, len(points), BATCH_SIZE):
    batch = points[start:start + BATCH_SIZE]

    if len(batch) < 2:
        continue  # OSRM match requires at least 2 points
    try:
        coordinates_str = ';'.join([f"{point.longitude},{point.latitude}" for point in batch])
        radiuses_str = ';'.join(['50'] * len(batch))
        yaw_rates_str = ';'.join([str(point.yaw_rate) for point in batch])
    except Exception as e:
        print(f"Error generating batch {start // BATCH_SIZE + 1}: {e}")
        continue
    osrm_match_url = (
        f"http://127.0.0.1:5001/match/v1/driving/{coordinates_str}"
        f"?overview=full&geometries=geojson&gaps=ignore&radiuses={radiuses_str}&yaw_rate={yaw_rates_str}"
    )

    response = requests.get(osrm_match_url)
    print(f"\nBatch {start // BATCH_SIZE + 1}: Requested URL:")
    print(osrm_match_url)
    print("Status Code:", response.status_code)

    if response.status_code != 200:
        print("Request failed. Skipping batch.")
        continue

    data = response.json()

    if data["code"] != "Ok" or "matchings" not in data or not data["matchings"]:
        print("Matching failed or no matchings returned.")
        continue

    # Plot matched coordinates (as line)
    for matching in data["matchings"]:
        matched_coords = matching["geometry"]["coordinates"]
        matched_coords = [[lat, lon] for lon, lat in matched_coords]  # Flip for folium
        folium.PolyLine(
            locations=matched_coords,
            color='green',
            weight=3,
        ).add_to(fmap)

    # Plot tracepoints (snapped points)
    for i, tp in enumerate(data.get("tracepoints", [])):
        if tp and tp.get("location"):
            lon, lat = tp["location"]
            folium.CircleMarker(
                location=[lat, lon],
                radius=2,
                color='blue',
                fill=True,
                popup=f"Snapped Point {start + i}"
            ).add_to(fmap)
    
    for p in batch:
        folium.CircleMarker(
            location=[p.latitude, p.longitude],
            radius=2,
            color='purple',
            fill=True,
            fill_color='purple'
        ).add_to(fmap)
        
    folium.PolyLine(
            locations=[(p.latitude, p.longitude) for p in batch],
            color='red',
            weight=3,
            opacity=.7
        ).add_to(fmap)
    
            
print("Total points:", len(points))
print("Any point without yaw_rate?", any(not hasattr(p, "yaw_rate") for p in points))
fmap


Total batches expected: 26

Batch 1: Requested URL:
http://127.0.0.1:5001/match/v1/driving/77.71111833,12.97371167;77.71111833,12.97371167;77.71111833,12.97371167;77.71111833,12.97371167;77.71111833,12.97371167;77.71111833,12.97371167;77.71111833,12.97371167;77.71111833,12.97371167;77.71111833,12.97371167;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,12.97371833;77.711115,1

In [8]:
matching['confidence']

0.9473181661